In [ ]:
# =============================================================================
# Radiation Treatment Planning — NSGA-II Multi-Objective Beam Angle Optimisation
# Patient: Lung_Patient_3  |  Dataset: PortPy (HuggingFace)
#
# KEY FEATURES:
#   - PlannerBeams.json loaded using the CONVEX code's direct approach:
#       beam_ids = positional indices (0-based) into the sorted beam_files list
#       (same logic as the convex notebook: beam_ids are used directly as array indices)
#   - LP sub-solver augmented with λ-regularisation: Frobenius + Group-lasso
#       (identical formulation to solve_convex_fast in the convex notebook)
#   - NSGA-II searches for best N_ANGLES beams within the planner candidate pool
#   - LAMBDA SWEEP: runs NSGA-II at multiple λ values and plots effect on
#       OAR score, PTV score, nuclear norm, hypervolume
#   - PLOTS: Pareto front, DVH, beamlet maps, lambda sweep, polar beam angles,
#       convergence, comparison bars, protocol pass/fail table
# =============================================================================

# ── SECTION 1: INSTALL ────────────────────────────────────────────────────────
# !pip install portpy cvxpy clarabel scipy matplotlib numpy h5py datasets
# !pip install huggingface_hub osqp

# ── SECTION 2: IMPORTS ────────────────────────────────────────────────────────
import os, json, warnings, time
from math import comb

import h5py
import numpy as np
import scipy.sparse as sp
from scipy.sparse import coo_matrix, hstack
import cvxpy as cp
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings('ignore')
matplotlib.rcParams["figure.dpi"] = 110

# ── Clinical dose limits (Gy) ─────────────────────────────────────────────────
D_PTV   = 60.0
D_ESOPH = 45.0
D_CORD  = 30.0
D_LUNG  = 20.0

# ── NSGA-II hyper-parameters ──────────────────────────────────────────────────
POP_SIZE      = 5
N_GENERATIONS = 10
MUTATION_RATE = 0.15
CROSSOVER_RATE= 0.9
SEED          = 42

# LP objective weights
W_PTV = 10.0
W_OAR = 2.0

# ── Lambda sweep values (for regularisation experiment) ───────────────────────
LAMBDA_VALUES = [0.0, 0.001, 0.005, 0.01, 0.05]   # run NSGA-II at each λ
REG_TYPE      = "both"   # "fro" | "group" | "both"  (matches convex notebook)

# OAR weights for NSGA-II objective 1
#fitness function
OAR_WEIGHTS = {'cord': 0.35, 'esoph': 0.30, 'lung': 0.35}

COLORS = ['#00c8ff', '#ff6b6b', '#a8ff78', '#ffd166']


# ── SECTION 3: DATA DOWNLOAD ──────────────────────────────────────────────────
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id      = "PortPy-Project/PortPy_Dataset",
    repo_type    = "dataset",
    allow_patterns = "data/Lung_Patient_3/*",
    local_dir    = "./hf_data"
)

PAT_DIR = './hf_data/data/Lung_Patient_3'


# ── SECTION 4: LOAD BEAMS — CONVEX-NOTEBOOK APPROACH ─────────────────────────
#
# The convex notebook treats PlannerBeams.json values as POSITIONAL INDICES
# (0-based) into the sorted beam_files list.  We replicate that exactly.
#
beam_dir   = os.path.join(PAT_DIR, 'Beams')
beam_files = sorted([
    os.path.join(beam_dir, f)
    for f in os.listdir(beam_dir)
    if f.endswith('_Data.h5')
])

# ── Read the voxel count from the first beam ──────────────────────────────────
with h5py.File(beam_files[0], 'r') as f:
    n_voxels = f['inf_matrix_full'].shape[0]

# ── Build the FULL dose influence matrix (all beams) ─────────────────────────
A_blocks_all   = []
bpb_all        = []   # beamlets-per-beam for every beam in the library

for bf in beam_files:
    with h5py.File(bf, 'r') as f:
        coo_data = f['inf_matrix_sparse'][:]
        n_cols   = f['inf_matrix_full'].shape[1]
        rows = coo_data[:, 0].astype(np.int32)
        cols = coo_data[:, 1].astype(np.int32)
        vals = coo_data[:, 2].astype(np.float32)
        A_beam = coo_matrix((vals, (rows, cols)),
                             shape=(n_voxels, n_cols)).tocsr()
        A_blocks_all.append(A_beam)
        bpb_all.append(n_cols)

A_all          = hstack(A_blocks_all, format='csr')
beam_col_all   = np.cumsum([0] + bpb_all)
n_beams_total  = len(beam_files)
print(f"[Library] {n_beams_total} beams, A shape: {A_all.shape}")

# ── Load PlannerBeams.json — same logic as convex notebook ───────────────────
planner_json_path = os.path.join(PAT_DIR, 'PlannerBeams.json')

with open(planner_json_path) as f:
    planner_raw = json.load(f)

# Convex notebook: beam_ids = positional indices into sorted beam_files
if isinstance(planner_raw, list):
    beam_ids = [int(x) for x in planner_raw]
elif isinstance(planner_raw, dict):
    # If the dict contains a list under a key, use that list
    key = next(
        (k for k in ('beam_ids', 'planner_beam_ids', 'IDs', 'ids') if k in planner_raw),
        None
    )
    if key is not None:
        beam_ids = [int(x) for x in planner_raw[key]]
    else:
        # Numeric-keyed dict: {"0": 3, "1": 7, ...} → values are positional indices
        beam_ids = [int(v) for v in planner_raw.values()]
else:
    raise ValueError(f"Unexpected PlannerBeams.json format: {type(planner_raw)}")

print(f"[Planner] Beam positional indices from PlannerBeams.json: {beam_ids}")
print(f"[Planner] Corresponding files:")
for i, b in enumerate(beam_ids):
    print(f"  [{i}] index={b}  →  {os.path.basename(beam_files[b])}")


# ── Build sub-matrices for planner beams (as in convex notebook) ─────────────
planner_col_indices = np.concatenate([
    np.arange(beam_col_all[b], beam_col_all[b + 1])
    for b in beam_ids
])

planner_bpb = [bpb_all[b] for b in beam_ids]
n_beams     = len(beam_ids)
max_bpb     = max(planner_bpb)
n_beamlets  = sum(planner_bpb)
beam_col_start = np.cumsum([0] + planner_bpb)

# N_ANGLES: how many beams NSGA-II selects from the planner pool
N_ANGLES = min(5, n_beams)

print(f"\n[Config] {n_beams} planner beams, selecting best {N_ANGLES}")
print(f"[Config] Total planner beamlets: {n_beamlets}")


# ── SECTION 5: LOAD VOXEL MASKS ───────────────────────────────────────────────
with h5py.File(os.path.join(PAT_DIR, 'OptimizationVoxels_Data.h5'), 'r') as f:
    ct_to_dose = f['ct_to_dose_voxel_map'][:]

with h5py.File(os.path.join(PAT_DIR, 'StructureSet_Data.h5'), 'r') as f:
    mask_ptv   = f['PTV'][:]
    mask_esoph = f['ESOPHAGUS'][:]
    mask_cord  = f['CORD'][:]
    mask_lung  = f['LUNG_L'][:]


def get_voxel_indices(mask_3d, ct_to_dose_map):
    both = mask_3d.astype(bool) & (ct_to_dose_map >= 0)
    return ct_to_dose_map[both].astype(np.int32)


ptv_voxels   = get_voxel_indices(mask_ptv,   ct_to_dose)
esoph_voxels = get_voxel_indices(mask_esoph, ct_to_dose)
cord_voxels  = get_voxel_indices(mask_cord,  ct_to_dose)
lung_voxels  = get_voxel_indices(mask_lung,  ct_to_dose)

# Full-library row subsets
A_ptv_all   = A_all[ptv_voxels,   :]
A_esoph_all = A_all[esoph_voxels, :]
A_cord_all  = A_all[cord_voxels,  :]
A_lung_all  = A_all[lung_voxels,  :]

# Planner-only column subsets (used in LP, matching convex notebook)
A_ptv_sp   = A_ptv_all[:,   planner_col_indices].astype(np.float64).tocsr()
A_esoph_sp = A_esoph_all[:, planner_col_indices].astype(np.float64).tocsr()
A_cord_sp  = A_cord_all[:,  planner_col_indices].astype(np.float64).tocsr()
A_lung_sp  = A_lung_all[:,  planner_col_indices].astype(np.float64).tocsr()

# Also keep full-planner versions for NSGA-II objective (beam-subset slicing)
A_ptv_full   = A_ptv_sp
A_esoph_full = A_esoph_sp
A_cord_full  = A_cord_sp
A_lung_full  = A_lung_sp

print(f"[Masks] PTV={len(ptv_voxels)}  Esoph={len(esoph_voxels)}  "
      f"Cord={len(cord_voxels)}  Lung={len(lung_voxels)} voxels")
print(f"[Matrix] Planner A shape: {A_ptv_sp.shape[1]} cols × {A_ptv_sp.shape[0]} ptv-rows (example)")


# ── SECTION 6: LOAD GANTRY ANGLES ────────────────────────────────────────────
gantry_angles = []
for b in beam_ids:
    bf = beam_files[b]
    with h5py.File(bf, 'r') as f:
        try:
            angle = float(f['gantry_angle'][()])
        except KeyError:
            try:
                angle = float(f['GantryAngle'][()])
            except KeyError:
                angle = None
    gantry_angles.append(angle)

if any(a is None for a in gantry_angles):
    print("[Angles] Metadata missing — using equally spaced 0–360°")
    gantry_angles = np.linspace(0, 360, n_beams, endpoint=False).tolist()

gantry_angles = np.array(gantry_angles, dtype=float)
print(f"[Angles] Planner gantry angles (°): {gantry_angles.astype(int).tolist()}")


# ── SECTION 7: CT VISUALISATION ───────────────────────────────────────────────
with h5py.File(os.path.join(PAT_DIR, 'CT_Data.h5'), 'r') as f:
    ct_hu = f[list(f.keys())[0]][:]

best_slice = np.argmax(mask_ptv.sum(axis=(1, 2)))
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(ct_hu[best_slice], cmap='gray', vmin=-1000, vmax=400)
for name, colour, mask in [
    ('PTV',       'red',    mask_ptv),
    ('ESOPHAGUS', 'yellow', mask_esoph),
    ('CORD',      'blue',   mask_cord),
    ('LUNG_L',    'green',  mask_lung),
]:
    ax.contour(mask[best_slice], levels=[0.5], colors=colour, linewidths=1.5)
    ax.plot([], [], color=colour, label=name)
ax.legend(loc='upper right', fontsize=10)
ax.set_title(f'CT Slice {best_slice} — Structure Contours')
ax.axis('off')
plt.tight_layout()
plt.savefig('ct_structures.png', dpi=150)
plt.show()
print("✓ ct_structures.png")


# ── SECTION 8: SAFE SVD HELPER ────────────────────────────────────────────────
def safe_svd(X_np):
    try:
        return np.linalg.svd(
            X_np + 1e-4 * np.random.randn(*X_np.shape), compute_uv=False)
    except np.linalg.LinAlgError:
        return np.array([np.linalg.norm(X_np, 'fro')])


# ── SECTION 9: REGULARISED LP SUB-SOLVER ─────────────────────────────────────
#
# Matches solve_convex_fast() from the convex notebook exactly.
# Adds λ * (Frobenius + Group-lasso) term to the LP objective.
# chromosome selects which of the n_beams planner beams are active.
#
_eval_cache: dict = {}


def solve_beam_subset(chromosome: np.ndarray,
                      lam: float = 0.0,
                      reg_type: str = "both") -> dict | None:
    """
    Solve the LP for a given beam subset chromosome.
    lam      : regularisation strength (0 = no reg)
    reg_type : 'fro' | 'group' | 'both'  (from convex notebook)
    """
    key = (tuple(chromosome), lam, reg_type)
    if key in _eval_cache:
        return _eval_cache[key]

    selected_beams = np.where(chromosome == 1)[0]
    col_indices    = np.concatenate([
        np.arange(beam_col_start[b], beam_col_start[b + 1])
        for b in selected_beams
    ])
    if len(col_indices) == 0:
        _eval_cache[key] = None
        return None

    Ap = A_ptv_full[:,   col_indices]
    Ae = A_esoph_full[:, col_indices]
    Ac = A_cord_full[:,  col_indices]
    Al = A_lung_full[:,  col_indices]

    n_b = len(col_indices)
    nP  = Ap.shape[0]
    nE  = Ae.shape[0]
    nC  = Ac.shape[0]
    nL  = Al.shape[0]

    # ── Build beam-matrix variable (for regularisation) ───────────────────────
    # Exactly mirrors the X_mat construction in solve_convex_fast
    x = cp.Variable(n_b, nonneg=True)

    sel_bpb    = [planner_bpb[b] for b in selected_beams]
    mbpb_sel   = max(sel_bpb)
    starts_sel = np.cumsum([0] + sel_bpb[:-1])

    rows_cp = []
    for i, sz in enumerate(sel_bpb):
        s = starts_sel[i]
        if sz < mbpb_sel:
            pad = cp.Constant(np.zeros(mbpb_sel - sz))
            row = cp.hstack([x[s:s + sz], pad])
        else:
            row = x[s:s + sz]
        rows_cp.append(row)
    X_mat = cp.vstack(rows_cp)   # shape: (n_active_beams, mbpb_sel)

    # ── Objective: clinical terms ─────────────────────────────────────────────
    obj_terms = [
        (W_PTV / nP) * cp.sum(cp.pos(D_PTV   - Ap @ x)),
        (W_OAR / nE) * cp.sum(cp.pos(Ae @ x  - D_ESOPH)),
        (W_OAR / nC) * cp.sum(cp.pos(Ac @ x  - D_CORD)),
    ]

    # ── Regularisation terms (Fro + Group-lasso) ─────────────────────────────
    if lam > 0:
        if reg_type == "fro":
            obj_terms.append(lam * cp.norm(X_mat, "fro"))
        elif reg_type == "group":
            obj_terms.append(lam * cp.sum(cp.norm(X_mat, axis=1)))
        elif reg_type == "both":
            # Combined: global shrinkage (Fro) + row sparsity (Group)
            obj_terms.append(lam * cp.norm(X_mat, "fro"))
            obj_terms.append(lam * cp.sum(cp.norm(X_mat, axis=1)))
        else:
            raise ValueError("reg_type must be 'fro', 'group', or 'both'")

    constraints = [cp.sum(Al @ x) / nL <= D_LUNG]
    prob = cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)

    for solver in [cp.OSQP, cp.ECOS, cp.SCS]:
        try:
            prob.solve(solver=solver, verbose=False,
                       **(dict(eps_abs=1e-4, eps_rel=1e-4, max_iter=20000)
                          if solver == cp.OSQP else {}))
            if prob.status in ('optimal', 'optimal_inaccurate') \
                    and x.value is not None:
                xv = np.maximum(x.value, 0)
                x_full = np.zeros(n_beamlets)
                x_full[col_indices] = xv
                result = {
                    'x_full':      x_full,
                    'col_indices': col_indices,
                    'solver':      str(solver),
                    'status':      prob.status,
                    'lam':         lam,
                    'reg_type':    reg_type,
                    'd_ptv':   np.asarray(A_ptv_full   @ x_full).ravel(),
                    'd_esoph': np.asarray(A_esoph_full @ x_full).ravel(),
                    'd_cord':  np.asarray(A_cord_full  @ x_full).ravel(),
                    'd_lung':  np.asarray(A_lung_full  @ x_full).ravel(),
                }
                _eval_cache[key] = result
                return result
        except Exception:
            continue

    _eval_cache[key] = None
    return None


# ── SECTION 10: OBJECTIVE FUNCTION ────────────────────────────────────────────
def compute_objectives(chromosome: np.ndarray,
                       lam: float = 0.0,
                       reg_type: str = "both") -> np.ndarray:
    sol = solve_beam_subset(chromosome, lam=lam, reg_type=reg_type)
    if sol is None:
        return np.array([1.0, 1.0])

    d95  = float(np.percentile(sol['d_ptv'], 5))
    obj2 = float(np.clip(1.0 - d95 / D_PTV, 0.0, 1.0))

    oar_score = (
        OAR_WEIGHTS['cord']  * sol['d_cord'].mean()  / D_PTV +
        OAR_WEIGHTS['esoph'] * sol['d_esoph'].mean() / D_PTV +
        OAR_WEIGHTS['lung']  * sol['d_lung'].mean()  / D_PTV
    )
    obj1 = float(np.clip(oar_score, 0.0, 1.0))
    return np.array([obj1, obj2])


# ── SECTION 11: NSGA-II OPERATORS ─────────────────────────────────────────────
def dominates(a, b):
    return np.all(a <= b) and np.any(a < b)


def fast_non_dominated_sort(objectives):
    N         = len(objectives)
    dom_count = np.zeros(N, dtype=int)
    dom_set   = [[] for _ in range(N)]
    fronts    = [[]]
    for i in range(N):
        for j in range(N):
            if i == j: continue
            if dominates(objectives[i], objectives[j]):
                dom_set[i].append(j)
            elif dominates(objectives[j], objectives[i]):
                dom_count[i] += 1
        if dom_count[i] == 0:
            fronts[0].append(i)
    k = 0
    while fronts[k]:
        nxt = []
        for i in fronts[k]:
            for j in dom_set[i]:
                dom_count[j] -= 1
                if dom_count[j] == 0:
                    nxt.append(j)
        k += 1
        fronts.append(nxt)
    return [f for f in fronts if f]


def crowding_distance(objectives, front):
    n = len(front)
    if n <= 2:
        return np.full(n, np.inf)
    dist = np.zeros(n)
    for m in range(objectives.shape[1]):
        vals  = objectives[front, m]
        order = np.argsort(vals)
        dist[order[0]] = dist[order[-1]] = np.inf
        rng = vals[order[-1]] - vals[order[0]]
        if rng == 0: continue
        for k in range(1, n - 1):
            dist[order[k]] += (vals[order[k + 1]] - vals[order[k - 1]]) / rng
    return dist


def random_chromosome():
    c = np.zeros(n_beams, dtype=int)
    c[np.random.choice(n_beams, N_ANGLES, replace=False)] = 1
    return c


def repair(c):
    c    = c.copy()
    diff = int(c.sum()) - N_ANGLES
    if diff > 0:
        c[np.random.choice(np.where(c == 1)[0], diff, replace=False)] = 0
    elif diff < 0:
        c[np.random.choice(np.where(c == 0)[0], -diff, replace=False)] = 1
    return c


def crossover(p1, p2):
    return repair(np.where(np.random.random(n_beams) < 0.5, p1, p2))


def mutate(c):
    if np.random.random() < MUTATION_RATE:
        on, off = np.where(c == 1)[0], np.where(c == 0)[0]
        if len(on) and len(off):
            c = c.copy()
            c[np.random.choice(on)]  = 0
            c[np.random.choice(off)] = 1
    return c


def tournament_selection(population, objectives, fronts, crowd, n_select):
    rank = np.zeros(len(population), dtype=int)
    for r, front in enumerate(fronts):
        for idx in front:
            rank[idx] = r
    selected, N = [], len(population)
    while len(selected) < n_select:
        i, j = np.random.choice(N, 2, replace=False)
        if   rank[i] < rank[j]:    selected.append(i)
        elif rank[j] < rank[i]:    selected.append(j)
        elif crowd[i] >= crowd[j]: selected.append(i)
        else:                      selected.append(j)
    return selected[:n_select]


def hypervolume_2d(pf_obj, ref=np.array([1.1, 1.1])):
    valid = pf_obj[(pf_obj[:, 0] < ref[0]) & (pf_obj[:, 1] < ref[1])]
    if not len(valid): return 0.0
    pts  = valid[np.argsort(valid[:, 0])]
    hv, prev = 0.0, ref[0]
    for i in range(len(pts) - 1, -1, -1):
        hv  += (prev - pts[i, 0]) * (ref[1] - pts[i, 1])
        prev = pts[i, 0]
    return hv


def knee_point(obj: np.ndarray) -> int:
    norm = obj.copy().astype(float)
    for m in range(norm.shape[1]):
        lo, hi = norm[:, m].min(), norm[:, m].max()
        if hi > lo:
            norm[:, m] = (norm[:, m] - lo) / (hi - lo)
    return int(np.argmin(np.linalg.norm(norm, axis=1)))


# ── SECTION 12: NSGA-II MAIN LOOP ────────────────────────────────────────────
def run_nsga2(lam: float = 0.0, reg_type: str = "both",
              seed: int = SEED) -> dict:
    """Run NSGA-II with given λ regularisation strength."""
    np.random.seed(seed)

    n_unique = comb(n_beams, N_ANGLES)
    if n_unique == 1:
      print("⚠️ Degenerate case: only one beam combination possible.")

      chrom = np.ones(n_beams, dtype=int)
      obj   = compute_objectives(chrom, lam=lam, reg_type=reg_type)

      return {
          'lam': lam,
          'reg_type': reg_type,
          'pf_pop': np.array([chrom]),
          'pf_obj': np.array([obj]),
          'knee_idx': 0,
          'history': {
              'gen': [1],
              'hv': [hypervolume_2d(np.array([obj]))],
              'pareto_size': [1],
              'best_oar': [obj[0]],
              'best_ptv': [obj[1]],
              'mean_oar': [obj[0]],
              'mean_ptv': [obj[1]],
          },
          'runtime': 0.0,
          'final_hv': hypervolume_2d(np.array([obj])),
      }

    eff_pop  = min(POP_SIZE, n_unique)

    print(f"\n{'='*65}")
    print(f"  NSGA-II  λ={lam}  reg={reg_type}")
    print(f"  Pool={n_beams} beams | Select={N_ANGLES} | "
          f"Pop={eff_pop} | Gens={N_GENERATIONS}")
    print(f"  Search space: C({n_beams},{N_ANGLES}) = {n_unique:,}")
    print(f"{'='*65}\n")

    seen, pop_list = set(), []
    while len(pop_list) < eff_pop:
        ind = random_chromosome()
        k   = tuple(ind)
        if k not in seen:
            seen.add(k); pop_list.append(ind)
    population = np.array(pop_list)

    print(f"  Evaluating initial population ({eff_pop} individuals)...")
    t0         = time.time()
    objectives = np.array([
        compute_objectives(ind, lam=lam, reg_type=reg_type)
        for ind in population
    ])
    print(f"  Init done in {time.time()-t0:.1f}s\n")

    history = {
        'gen': [], 'hv': [], 'pareto_size': [],
        'best_oar': [], 'best_ptv': [],
        'mean_oar': [], 'mean_ptv': [],
    }
    t_run = time.time()

    for gen in range(N_GENERATIONS):
        fronts = fast_non_dominated_sort(objectives)
        crowd  = np.zeros(eff_pop)
        for front in fronts:
            cd = crowding_distance(objectives, front)
            for k_idx, idx in enumerate(front):
                crowd[idx] = cd[k_idx]

        pf_idx = fronts[0]
        pf_obj = objectives[pf_idx]
        hv     = hypervolume_2d(pf_obj)

        history['gen'].append(gen + 1)
        history['hv'].append(hv)
        history['pareto_size'].append(len(pf_idx))
        history['best_oar'].append(float(pf_obj[:, 0].min()))
        history['best_ptv'].append(float(pf_obj[:, 1].min()))
        history['mean_oar'].append(float(objectives[:, 0].mean()))
        history['mean_ptv'].append(float(objectives[:, 1].mean()))

        print(f"  Gen {gen+1}/{N_GENERATIONS}  "
              f"HV={hv:.5f}  PF={len(pf_idx)}  "
              f"Best OAR={pf_obj[:,0].min():.4f}  "
              f"Best PTV={pf_obj[:,1].min():.4f}  "
              f"[{time.time()-t_run:.1f}s]")

        parents = [population[i] for i in
                   tournament_selection(population, objectives,
                                        fronts, crowd, eff_pop)]
        np.random.shuffle(parents)
        offspring = []
        for k in range(0, eff_pop, 2):
            p1, p2 = parents[k], parents[min(k + 1, eff_pop - 1)]
            c1 = crossover(p1, p2) if np.random.random() < CROSSOVER_RATE else p1.copy()
            c2 = crossover(p2, p1) if np.random.random() < CROSSOVER_RATE else p2.copy()
            offspring += [mutate(c1), mutate(c2)]
        offspring = np.array(offspring[:eff_pop])
        off_obj   = np.array([
            compute_objectives(ind, lam=lam, reg_type=reg_type)
            for ind in offspring
        ])

        comb_pop = np.vstack([population, offspring])
        comb_obj = np.vstack([objectives, off_obj])

        c_fronts = fast_non_dominated_sort(comb_obj)
        next_idx = []
        for front in c_fronts:
            if len(next_idx) + len(front) <= eff_pop:
                next_idx.extend(front)
            else:
                needed  = eff_pop - len(next_idx)
                cd_vals = crowding_distance(comb_obj, front)
                ranked  = sorted(range(len(front)),
                                 key=lambda k: cd_vals[k], reverse=True)
                next_idx.extend(front[ranked[i]] for i in range(needed))
                break

        population = comb_pop[next_idx]
        objectives = comb_obj[next_idx]

    runtime = time.time() - t_run
    pf_idx  = fast_non_dominated_sort(objectives)[0]
    pf_pop  = population[pf_idx]
    pf_obj  = objectives[pf_idx]
    order   = np.argsort(pf_obj[:, 0])
    pf_pop, pf_obj = pf_pop[order], pf_obj[order]
    ki       = knee_point(pf_obj)
    final_hv = hypervolume_2d(pf_obj)

    print(f"\n  DONE  λ={lam}  HV={final_hv:.5f}  "
          f"Knee: OAR={pf_obj[ki,0]:.4f} PTV={pf_obj[ki,1]:.4f}  "
          f"Runtime={runtime:.1f}s\n")

    return {
        'lam':      lam,
        'reg_type': reg_type,
        'pf_pop':   pf_pop,
        'pf_obj':   pf_obj,
        'knee_idx': ki,
        'history':  history,
        'runtime':  runtime,
        'final_hv': final_hv,
    }


# ── SECTION 13: METRICS ───────────────────────────────────────────────────────
def compute_metrics(chrom, sol, obj, label):
    if sol is None:
        return {'label': label, 'status': 'failed'}
    xv = sol['x_full']
    starts = np.cumsum([0] + planner_bpb)
    mbpb   = max(planner_bpb)
    X_np   = np.zeros((n_beams, mbpb))
    for i in range(n_beams):
        s, sz = starts[i], planner_bpb[i]
        X_np[i, :sz] = xv[s:s + sz]
    svs = safe_svd(X_np)

    sel = [f"{int(gantry_angles[i])}°" for i, v in enumerate(chrom) if v == 1]

    return {
        'label'        : label,
        'status'       : sol.get('status', 'optimal'),
        'solver'       : sol.get('solver', '?'),
        'lam'          : sol.get('lam', 0.0),
        'OAR_score'    : round(float(obj[0]), 4),
        'PTV_score'    : round(float(obj[1]), 4),
        'D95_ptv'      : round(float(np.percentile(sol['d_ptv'],  5)), 2),
        'D05_ptv'      : round(float(np.percentile(sol['d_ptv'], 95)), 2),
        'Dmean_ptv'    : round(float(sol['d_ptv'].mean()), 2),
        'HI'           : round(float((np.percentile(sol['d_ptv'], 95) -
                                      np.percentile(sol['d_ptv'],  5)) / D_PTV), 4),
        'CI_proxy_%'   : round(float(np.mean(sol['d_ptv'] >= 0.95 * D_PTV) * 100), 2),
        'F1_underdose' : round(float(np.mean(np.maximum(D_PTV - sol['d_ptv'], 0))), 4),
        'Dmean_esoph'  : round(float(sol['d_esoph'].mean()), 2),
        'Dmax_esoph'   : round(float(sol['d_esoph'].max()),  2),
        'Dmean_cord'   : round(float(sol['d_cord'].mean()), 2),
        'Dmax_cord'    : round(float(sol['d_cord'].max()),  2),
        'Dmean_lung'   : round(float(sol['d_lung'].mean()), 2),
        'V20_lung_%'   : round(float(np.mean(sol['d_lung'] >= 20.0) * 100), 2),
        'F2_overdose'  : round(float(
                           np.mean(np.maximum(sol['d_esoph'] - D_ESOPH, 0)) +
                           np.mean(np.maximum(sol['d_cord']  - D_CORD,  0))), 4),
        'lung_viol'    : round(float(max(sol['d_lung'].mean() - D_LUNG, 0)), 4),
        'sparsity_%'   : round(float(np.mean(xv < 1e-3) * 100), 2),
        'total_MU'     : round(float(xv.sum()), 2),
        'nuc_norm'     : round(float(svs.sum()), 4),
        'eff_rank'     : round(float((svs.sum()**2) / ((svs**2).sum() + 1e-12)), 3),
        'angles_deg'   : str(sel),
    }


# ── SECTION 14: PROTOCOL PASS/FAIL ────────────────────────────────────────────
def print_validation_table(df, tag="PROTOCOL PASS/FAIL"):
    limits = {
        'D95_ptv'    : ('>=', 57.0, 'PTV coverage'),
        'HI'         : ('<=', 0.10, 'PTV homogeneity'),
        'Dmax_esoph' : ('<=', 45.0, 'Esoph sparing'),
        'Dmax_cord'  : ('<=', 30.0, 'Cord sparing'),
        'Dmean_lung' : ('<=', 20.0, 'Lung mean dose'),
    }
    print(f"\n{'='*72}\n{tag:^72}\n{'='*72}")
    header = f"  {'Metric':<18} {'Limit':<10}" + \
             "".join(f"{l:>16}" for l in df.index)
    print(header); print('-' * 72)
    for metric, (op, limit, desc) in limits.items():
        if metric not in df.columns: continue
        vals = df[metric].astype(float)
        def flag(v): return ' ✓' if (v >= limit if op == '>=' else v <= limit) else ' ✗'
        row = f"  {desc:<18} {op}{limit:<9}" + \
              "".join(f"{str(vals[l])+flag(vals[l]):>16}" for l in df.index)
        print(row)
    print('='*72 + "\n")


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 15: RUN LAMBDA SWEEP
# Run NSGA-II independently at each λ value.
# The cache is SHARED across runs, so re-evaluating the same chromosome
# (with the same λ) costs nothing.
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  LAMBDA SWEEP — Running NSGA-II at each λ value")
print(f"  λ values: {LAMBDA_VALUES}  reg_type: {REG_TYPE}")
print("═"*65 + "\n")

SWEEP_RESULTS = {}   # lam → run_nsga2() output

for lam_val in LAMBDA_VALUES:
    _eval_cache = {}   # clear per-λ run (different regularisation = different LP)
    result = run_nsga2(lam=lam_val, reg_type=REG_TYPE, seed=SEED)
    SWEEP_RESULTS[lam_val] = result

# ── Use the last run (highest λ) as the primary result for detailed plots ──────
# (user can change this to any λ)
PRIMARY_LAM = LAMBDA_VALUES[0]   # baseline: no regularisation
PRIMARY_RESULT = SWEEP_RESULTS[PRIMARY_LAM]

pf_pop  = PRIMARY_RESULT['pf_pop']
pf_obj  = PRIMARY_RESULT['pf_obj']
ki      = PRIMARY_RESULT['knee_idx']
history = PRIMARY_RESULT['history']

# Re-solve for primary run's Pareto solutions (needed for plots)
_eval_cache = {}   # reset cache for clean re-solve
print("  Re-solving LP for primary Pareto solutions...")
pareto_solutions = [
    solve_beam_subset(chrom, lam=PRIMARY_LAM, reg_type=REG_TYPE)
    for chrom in pf_pop
]
knee_sol = pareto_solutions[ki]

# Showcase: Best OAR, Knee, Best PTV
showcase_idx = sorted(set([0, ki, len(pf_pop) - 1]))
SHOWCASE = {}
for idx in showcase_idx:
    tag = ('Best OAR' if idx == 0 else
           'Knee ★'   if idx == ki else
           'Best PTV')
    SHOWCASE[tag] = {
        'sol':   pareto_solutions[idx],
        'chrom': pf_pop[idx],
        'obj':   pf_obj[idx],
    }

rows = [compute_metrics(v['chrom'], v['sol'], v['obj'], k)
        for k, v in SHOWCASE.items()]
df   = pd.DataFrame(rows).set_index('label')

print_validation_table(df)


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 16: PLOTS
# ══════════════════════════════════════════════════════════════════════════════

# ── Plot 1: Pareto Front ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5.5), facecolor='#0e1117')
ax.set_facecolor('#0e1117')
ax.plot(pf_obj[:, 0], pf_obj[:, 1], color='#00c8ff', lw=1.2, alpha=0.4)
ax.scatter(pf_obj[:, 0], pf_obj[:, 1],
           color='#00c8ff', s=45, edgecolors='white', lw=0.4,
           label='Pareto solutions')
ax.scatter(pf_obj[ki, 0], pf_obj[ki, 1],
           color='gold', s=130, edgecolors='k', lw=0.8,
           zorder=5, label='Knee point ★')
for idx, tag in [(0, 'Best OAR'), (len(pf_pop)-1, 'Best PTV')]:
    ax.annotate(tag,
                xy=(pf_obj[idx, 0], pf_obj[idx, 1]),
                xytext=(pf_obj[idx, 0] + 0.025, pf_obj[idx, 1] + 0.025),
                color='white', fontsize=8,
                arrowprops=dict(arrowstyle='->', color='white', lw=0.7))
ax.set_xlabel('obj1 — Weighted OAR dose score ↓', color='white', fontsize=10)
ax.set_ylabel('obj2 — PTV undercoverage ↓',        color='white', fontsize=10)
ax.set_title(f'NSGA-II Pareto Front (λ={PRIMARY_LAM}, reg={REG_TYPE})\n'
             f'HV={PRIMARY_RESULT["final_hv"]:.4f}  |  '
             f'{len(pf_pop)} non-dominated solutions',
             color='white', fontsize=11)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')
ax.grid(color='#2a2a2a', linewidth=0.5)
ax.legend(facecolor='#1c1c2e', labelcolor='white', fontsize=9)
plt.tight_layout()
plt.savefig('pareto_front.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()
print("✓ pareto_front.png")


# ── Plot 2: DVH ───────────────────────────────────────────────────────────────
def plot_dvh(showcase_dict, dose_max=80.0, n_bins=300):
    bins    = np.linspace(0, dose_max, n_bins)
    structs = [
        ('PTV',   'd_ptv',   D_PTV,   '-',               '#ffffff'),
        ('Esoph', 'd_esoph', D_ESOPH, '--',               '#ffd166'),
        ('Cord',  'd_cord',  D_CORD,  '-.',               '#ff9f43'),
        ('Lung',  'd_lung',  D_LUNG,  (0,(3,1,1,1,1,1)), '#a29bfe'),
    ]
    n_res = len(showcase_dict)
    fig, axes = plt.subplots(1, n_res, figsize=(7 * n_res, 5), sharey=True,
                             facecolor='#0e1117')
    if n_res == 1: axes = [axes]
    for ax, (label, data), col in zip(axes, showcase_dict.items(), COLORS):
        sol = data['sol']
        if sol is None:
            ax.set_title(f'{label}\n(no solution)', color='white'); continue
        for sname, dkey, limit, ls, sc in structs:
            dose = sol[dkey]
            dvh  = np.array([(dose >= d).mean() * 100 for d in bins])
            ax.plot(bins, dvh, color=sc, linestyle=ls, linewidth=2, label=sname)
            ax.axvline(limit, color=sc, linewidth=0.6, linestyle=':', alpha=0.45)
        d95 = np.percentile(sol['d_ptv'], 5)
        ax.annotate(f'D95={d95:.1f}Gy', xy=(d95, 95), xytext=(d95 + 3, 85),
                    color='white', fontsize=8,
                    arrowprops=dict(arrowstyle='->', color='white', lw=0.8))
        ax.set_facecolor('#0e1117')
        ax.set_title(f'{label}\nOAR={data["obj"][0]:.3f}  PTV={data["obj"][1]:.3f}',
                     color='white', fontsize=10)
        ax.set_xlabel('Dose (Gy)', color='white')
        if ax is axes[0]: ax.set_ylabel('Volume (%)', color='white')
        ax.tick_params(colors='white')
        ax.set_xlim(0, dose_max); ax.set_ylim(0, 105)
        for sp in ax.spines.values(): sp.set_edgecolor('#444')
        ax.grid(color='#2a2a2a', linewidth=0.5)
        ax.legend(facecolor='#1c1c2e', labelcolor='white', fontsize=9)
    plt.suptitle(f'DVH — NSGA-II Selected Plans (λ={PRIMARY_LAM}, reg={REG_TYPE})',
                 color='white', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('dvh.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()
    print("✓ dvh.png")


plot_dvh(SHOWCASE)


# ── Plot 3: Beamlet Intensity Maps ────────────────────────────────────────────
def plot_beamlet_maps(showcase_dict, cmap='inferno'):
    n_res = len(showcase_dict)
    fig, axes = plt.subplots(n_res, n_beams,
                             figsize=(n_beams * 2.0, n_res * 2.5),
                             facecolor='#0e1117', squeeze=False)
    for row, (label, data) in enumerate(showcase_dict.items()):
        sol = data['sol']
        if sol is None: continue
        xv   = sol['x_full']
        vmax = max(xv[beam_col_start[b]:beam_col_start[b+1]].max() + 1e-9
                   for b in range(n_beams))
        for col in range(n_beams):
            ax  = axes[row, col]
            s, e = beam_col_start[col], beam_col_start[col + 1]
            beam_x = xv[s:e]; sz = e - s
            side   = int(np.ceil(np.sqrt(sz)))
            padded = np.zeros(side * side); padded[:sz] = beam_x
            im = ax.imshow(padded.reshape(side, side), cmap=cmap,
                           aspect='auto', vmin=0, vmax=vmax,
                           interpolation='nearest')
            is_on = data['chrom'][col] == 1
            ax.set_title(f'{int(gantry_angles[col])}°',
                         color='white' if is_on else '#555', fontsize=6)
            ax.axis('off')
            if col == n_beams - 1:
                cb = plt.colorbar(im, ax=ax, fraction=0.05, pad=0.04)
                cb.ax.tick_params(colors='white', labelsize=6)
        axes[row, 0].set_ylabel(label, color='white', fontsize=9)
    plt.suptitle(f'Beamlet Intensity Maps — λ={PRIMARY_LAM}, reg={REG_TYPE}\n'
                 '(dimmed = not chosen by NSGA-II)',
                 color='white', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('beamlet_maps.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()
    print("✓ beamlet_maps.png")


plot_beamlet_maps(SHOWCASE)


# ── Plot 4: Lambda Sweep — Effect on OAR, PTV, Nuclear Norm, HV ───────────────
def plot_lambda_sweep(sweep_results):
    """
    For each λ, take the knee point solution and extract:
      - best OAR score, best PTV score, HV, nuclear norm
    This mirrors the convex notebook's lambda_sweep plot but adds HV.
    """
    lams_valid, oar_vals, ptv_vals, hv_vals, nuc_vals = [], [], [], [], []
    f1_vals, f2_vals = [], []

    for lam_val, res in sweep_results.items():
        pf = res['pf_obj']
        ki_l = res['knee_idx']
        knee_chrom = res['pf_pop'][ki_l]

        # Re-solve knee solution for this λ if not cached
        sol = solve_beam_subset(knee_chrom, lam=lam_val, reg_type=res['reg_type'])
        if sol is None:
            continue

        xv = sol['x_full']
        starts = np.cumsum([0] + planner_bpb)
        mbpb   = max(planner_bpb)
        X_np   = np.zeros((n_beams, mbpb))
        for i in range(n_beams):
            s, sz = starts[i], planner_bpb[i]
            X_np[i, :sz] = xv[s:s + sz]
        svs = safe_svd(X_np)

        lams_valid.append(lam_val)
        oar_vals.append(float(pf[ki_l, 0]))
        ptv_vals.append(float(pf[ki_l, 1]))
        hv_vals.append(float(res['final_hv']))
        nuc_vals.append(float(svs.sum()))
        f1_vals.append(float(np.mean(np.maximum(D_PTV - sol['d_ptv'], 0))))
        f2_vals.append(float(
            np.mean(np.maximum(sol['d_esoph'] - D_ESOPH, 0)) +
            np.mean(np.maximum(sol['d_cord']  - D_CORD,  0))
        ))

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor='#0e1117')
    axes = axes.flatten()

    panels = [
        (lams_valid, oar_vals,  'OAR score (knee) ↓',          '#00c8ff'),
        (lams_valid, ptv_vals,  'PTV undercoverage (knee) ↓',   '#ff6b6b'),
        (lams_valid, hv_vals,   'Hypervolume ↑',                '#a8ff78'),
        (lams_valid, nuc_vals,  'Nuclear norm (knee) ↓',        '#ffd166'),
        (lams_valid, f1_vals,   'F1 under-dose (lower=better)', '#a29bfe'),
        (lams_valid, f2_vals,   'F2 OAR over-dose (lower=better)','#ff9f43'),
    ]
    for ax, (xs, ys, ylabel, col) in zip(axes, panels):
        ax.plot(xs, ys, 'o-', color=col, linewidth=2, markersize=7)
        ax.fill_between(xs, min(ys)*0.98, ys, color=col, alpha=0.10)
        ax.set_facecolor('#0e1117')
        ax.set_xlabel(f'λ ({REG_TYPE} regularisation)', color='white', fontsize=9)
        ax.set_ylabel(ylabel, color='white', fontsize=9)
        ax.tick_params(colors='white')
        for sp in ax.spines.values(): sp.set_edgecolor('#444')
        ax.grid(color='#2a2a2a', linewidth=0.5)
        # Mark baseline
        if 0.0 in xs:
            bi = xs.index(0.0)
            ax.axvline(xs[bi], color='white', linewidth=0.6, linestyle=':', alpha=0.5)
            ax.text(xs[bi], min(ys), ' baseline', color='white', fontsize=7, alpha=0.7)

    plt.suptitle(f'λ-Regularisation Sweep (Fro+Group) on NSGA-II Knee Point\n'
                 f'n_beams={n_beams}, N_ANGLES={N_ANGLES}, '
                 f'Pop={POP_SIZE}, Gens={N_GENERATIONS}',
                 color='white', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('lambda_sweep.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()
    print("✓ lambda_sweep.png")


plot_lambda_sweep(SWEEP_RESULTS)


# ── Plot 5: Multi-λ Pareto Fronts Overlay ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6), facecolor='#0e1117')
ax.set_facecolor('#0e1117')
lam_colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(LAMBDA_VALUES)))

for (lam_val, res), col in zip(SWEEP_RESULTS.items(), lam_colors):
    pf = res['pf_obj']
    ki_l = res['knee_idx']
    order = np.argsort(pf[:, 0])
    ax.plot(pf[order, 0], pf[order, 1], color=col, lw=1.5, alpha=0.7,
            label=f'λ={lam_val}')
    ax.scatter(pf[:, 0], pf[:, 1], color=col, s=25, alpha=0.5)
    ax.scatter(pf[ki_l, 0], pf[ki_l, 1], color=col,
               s=100, edgecolors='white', lw=0.8, zorder=5)

ax.set_xlabel('obj1 — Weighted OAR score ↓', color='white', fontsize=10)
ax.set_ylabel('obj2 — PTV undercoverage ↓',   color='white', fontsize=10)
ax.set_title(f'Pareto Fronts at Different λ Values (★ = knee)\n'
             f'reg_type={REG_TYPE}', color='white', fontsize=11)
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_edgecolor('#444')
ax.grid(color='#2a2a2a', linewidth=0.5)
ax.legend(facecolor='#1c1c2e', labelcolor='white', fontsize=9)
plt.tight_layout()
plt.savefig('pareto_all_lambdas.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()
print("✓ pareto_all_lambdas.png")


# ── Plot 6: HV Convergence ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor='#0e1117')
h = history
panels_hv = [
    (h['hv'],       'Hypervolume ↑',            '#a8ff78'),
    (h['best_oar'], 'Best OAR score ↓',         '#00c8ff'),
    (h['best_ptv'], 'Best PTV undercoverage ↓', '#ff6b6b'),
]
for ax, (vals, ylabel, col) in zip(axes, panels_hv):
    ax.set_facecolor('#0e1117')
    ax.plot(h['gen'], vals, color=col, lw=2.0)
    ax.fill_between(h['gen'], 0, vals, color=col, alpha=0.12)
    ax.set_xlabel('Generation', color='white', fontsize=9)
    ax.set_ylabel(ylabel,        color='white', fontsize=9)
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
    ax.grid(color='#2a2a2a', linewidth=0.5)
plt.suptitle(f'NSGA-II Convergence (λ={PRIMARY_LAM}, reg={REG_TYPE})',
             color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('hv_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()
print("✓ hv_convergence.png")


# ── Plot 7: Polar Beam Angle Plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, len(SHOWCASE),
                          figsize=(4.5 * len(SHOWCASE), 4.5),
                          subplot_kw={'projection': 'polar'},
                          facecolor='#0e1117')
if len(SHOWCASE) == 1: axes = [axes]
for ax, (label, data), col in zip(axes, SHOWCASE.items(), COLORS):
    chrom = data['chrom']
    for a in gantry_angles:
        ax.plot([0, np.deg2rad(a)], [0, 1], color='#333', lw=0.5, alpha=0.6)
    for i, (v, a) in enumerate(zip(chrom, gantry_angles)):
        if v == 1:
            ax.annotate("", xy=(np.deg2rad(a), 1.0), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='->', color=col, lw=2.0))
    ax.set_facecolor('#0e1117')
    ax.set_yticklabels([])
    ax.set_xticks(np.deg2rad([0,45,90,135,180,225,270,315]))
    ax.set_xticklabels(['0°','45°','90°','135°','180°','225°','270°','315°'],
                        color='white', fontsize=7)
    ax.set_title(label, color='white', fontsize=10, pad=10)
    for sp in ax.spines.values(): sp.set_edgecolor('#444')
plt.suptitle(f'Selected Beam Angles — λ={PRIMARY_LAM}',
             color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('beam_angles_polar.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
plt.show()
print("✓ beam_angles_polar.png")


# ── Plot 8: Comparison Bar Charts ─────────────────────────────────────────────
def plot_comparison_bars(df_metrics):
    metric_groups = {
        'Dose Quality'   : ['F1_underdose', 'F2_overdose', 'HI', 'lung_viol'],
        'Plan Structure' : ['nuc_norm', 'eff_rank', 'sparsity_%'],
    }
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0e1117')
    labels = df_metrics.index.tolist()
    x_base = np.arange(len(labels))
    for ax, (grp_name, metrics) in zip(axes, metric_groups.items()):
        n_m   = len(metrics)
        width = 0.7 / n_m
        ax.set_facecolor('#0e1117')
        for i, metric in enumerate(metrics):
            if metric not in df_metrics.columns: continue
            vals = df_metrics[metric].astype(float).values
            col  = COLORS[i % len(COLORS)]
            bars = ax.bar(x_base + i * width - (n_m - 1) * width / 2,
                          vals, width * 0.88, label=metric, color=col, alpha=0.85)
            for bar, val in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2,
                        bar.get_height() + 0.002,
                        f'{val:.3f}', ha='center', va='bottom',
                        color='white', fontsize=7.5, rotation=40)
        ax.set_xticks(x_base)
        ax.set_xticklabels(labels, color='white', fontsize=9, rotation=10)
        ax.tick_params(colors='white')
        ax.set_title(grp_name, color='white', fontsize=11)
        ax.set_ylabel('Value', color='white', fontsize=9)
        for sp in ax.spines.values(): sp.set_edgecolor('#444')
        ax.legend(facecolor='#1c1c2e', labelcolor='white', fontsize=8)
        ax.grid(axis='y', color='#2a2a2a', linewidth=0.5)
    plt.suptitle(f'Plan Quality Comparison — NSGA-II (λ={PRIMARY_LAM}, reg={REG_TYPE})',
                 color='white', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig('comparison_bars.png', dpi=150, bbox_inches='tight', facecolor='#0e1117')
    plt.show()
    print("✓ comparison_bars.png")


plot_comparison_bars(df)


# ── Full Metrics Table ────────────────────────────────────────────────────────
groups = {
    'NSGA-II Scores' : ['OAR_score', 'PTV_score', 'lam'],
    'PTV Coverage'   : ['D95_ptv','D05_ptv','Dmean_ptv','HI',
                        'CI_proxy_%','F1_underdose'],
    'OAR Sparing'    : ['Dmean_esoph','Dmax_esoph','Dmean_cord','Dmax_cord',
                        'Dmean_lung','V20_lung_%','F2_overdose','lung_viol'],
    'Plan Quality'   : ['sparsity_%','total_MU','nuc_norm','eff_rank'],
    'Beam Selection' : ['angles_deg'],
}
col_w  = max(18, max(len(l) for l in df.index) + 2)
header = f"{'METRIC':<24}" + "".join(f"{l:>{col_w}}" for l in df.index)
print('\n' + '═' * len(header))
print(header)
print('═' * len(header))
for grp, keys in groups.items():
    print(f'\n  ── {grp} ──')
    for k in keys:
        if k not in df.columns: continue
        row = "".join(f"{str(df.loc[l, k]):>{col_w}}" for l in df.index)
        print(f"  {k:<24}{row}")
print('═' * len(header))

# ── Lambda sweep summary table ────────────────────────────────────────────────
print(f"\n{'═'*65}")
print(f"{'LAMBDA SWEEP SUMMARY':^65}")
print(f"{'═'*65}")
print(f"  {'λ':>8}  {'HV':>10}  {'PF_size':>8}  {'Best_OAR':>10}  {'Best_PTV':>10}")
print('-' * 55)
for lam_val, res in SWEEP_RESULTS.items():
    print(f"  {lam_val:>8.4f}  "
          f"{res['final_hv']:>10.5f}  "
          f"{len(res['pf_pop']):>8}  "
          f"{res['pf_obj'][:,0].min():>10.4f}  "
          f"{res['pf_obj'][:,1].min():>10.4f}")
print('═' * 65)

print("\n✓ All outputs saved.")